In [209]:
from bs4 import BeautifulSoup
import urllib.request
import pandas as pd
import datetime
import requests
from selenium import webdriver
from selenium.webdriver.common.by import By
import time
import random
import matplotlib as mp
import numpy as np
import seaborn as sns
import math

In [2]:
titanic = sns.load_dataset("titanic")

In [3]:
#구간 정의
bins = [-1, 0, 2, 8]
#라벨링
labels = ['0명', '1~2명', '3명 이상']
#pd.cut 사용해서 구하기
titanic['sibsp_group']=pd.cut(titanic['sibsp'], bins, labels=labels)
#groupby로 묶어주고 3개 컬럼 계산하기
dif = titanic.groupby('sibsp_group')[["survived","age","fare"]].mean()
dif

C:\Users\smart\AppData\Local\Temp\ipykernel_17456\3548481487.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  dif = titanic.groupby('sibsp_group')[["survived","age","fare"]].mean()


,survived,age,fare
sibsp_group,,,
0명,0.345395,31.397558,25.692028
1~2명,0.527426,29.191923,45.046011
3명 이상,0.152174,9.857143,52.115039


In [53]:
tips = sns.load_dataset("tips")
tips.head()
tips.tail()

,total_bill,tip,sex,smoker,day,time,size
239,29.03,5.92,Male,No,Sat,Dinner,3
240,27.18,2.00,Female,Yes,Sat,Dinner,2
241,22.67,2.00,Male,Yes,Sat,Dinner,2
242,17.82,1.75,Male,No,Sat,Dinner,2
243,18.78,3.00,Female,No,Thur,Dinner,2


In [54]:
#1번
tips["tip_pct"] = round(tips['tip'] / tips['total_bill'] *100 ,2)
tips["sex/time"] = tips['sex'].astype(str)+tips['time'].astype(str)
tips.groupby('sex/time')[['total_bill','tip','tip_pct']].mean()

#일단 기본적으로 팁을 주는 가격의 마지노선이 정해져있다
#남자가 더 많이 먹어서 팁도 조금 더 주지만 총 가격이 높기에 비율은 어느정도 비슷하다.
#팁의 최저 마지노선이 있기에 적게 먹어도 최저만큼은 줘야해서 여성의 팁 비율이 조금 높은듯하다.


,total_bill,tip,tip_pct
sex/time,,,
FemaleDinner,19.213077,3.002115,16.931731
FemaleLunch,16.339143,2.582857,16.227143
MaleDinner,21.461452,3.144839,15.540403
MaleLunch,18.048485,2.882121,16.607576


In [100]:
#1번
tips["tip_pct"] = round(tips['tip'] / tips['total_bill'] *100 ,2)
# tips["sex/time"] = tips['sex'].astype(str)+tips['time'].astype(str)
tips.groupby(['sex','time'], observed=False)[['tip_pct']].mean().sort_values('tip_pct',ascending=False)

#일단 기본적으로 팁을 주는 가격의 마지노선이 정해져있다
#남자가 더 많이 먹어서 팁도 조금 더 주지만 총 가격이 높기에 비율은 어느정도 비슷하다.
#팁의 최저 마지노선이 있기에 적게 먹어도 최저만큼은 줘야해서 여성의 팁 비율이 조금 높은듯하다.


,,tip_pct
sex,time,
Female,Dinner,16.931731
Male,Lunch,16.607576
Female,Lunch,16.227143
Male,Dinner,15.540403


In [ ]:
tips['tip_rank'] = tips['tip_pct'].rank(pct=True)
#quantile

In [ ]:
#2번
tips['tip_rank'] = tips['tip_pct'].rank(pct=True)
bins = [0, 0.2, 0.5, 1.0]
labels = ["top-tier","mid-tier",'low-tier']
tips['tip_level'] = pd.cut(tips['tip_rank'],bins=bins,labels=labels)
tips.groupby(['sex','time','tip_level'],observed=False)['tip_pct'].mean().rank().sort_values().head(3)
#여자들은 조금먹지만 팁이 너무 적으면 안되어서 먹은거 대비 팁을 많이준다
#남자들은 많이 먹고 팁도 많이 준다.

sex     time    tip_level
Female  Dinner  top-tier     1.0
Male    Dinner  top-tier     2.0
Female  Lunch   top-tier     3.0
Name: tip_pct, dtype: float64

In [89]:
#3번
total = tips.groupby(['day','time','sex','smoker'],observed=False)['tip'].agg(['mean','var','count']).reset_index()
total['mean_rank'] = total['mean'].rank(method = 'min')
total['var_rank'] = total['var'].rank(method = 'min')
total['rank'] = total['mean_rank'] + total['var_rank']
total['rank'] = total['rank'].rank(method='min')
total.sort_values('rank')
#금요일 점심에 담배피는 남자들은 내일이 주말이라 기분이 좋아서 팁을 많이준다
#금요일 점심에는 여자도 기분이 좋기에 팁을 많이준다
#목요일 점심에 여자들은 왜 팁을 많이줄까 모르겠다

,day,time,sex,smoker,mean,var,count,mean_rank,var_rank,rank
8,Fri,Lunch,Male,Yes,1.900000,0.096400,3,1.0,1.0,1.0
3,Thur,Lunch,Female,No,2.437083,1.200213,24,2.0,5.0,2.0
10,Fri,Lunch,Female,Yes,2.660000,0.566800,3,4.0,3.0,2.0
23,Sat,Dinner,Female,No,2.724615,0.925260,13,6.0,4.0,4.0
13,Fri,Dinner,Male,No,2.500000,2.000000,2,3.0,12.0,5.0
14,Fri,Dinner,Female,Yes,2.700000,1.860000,4,5.0,11.0,6.0
2,Thur,Lunch,Female,Yes,2.990000,1.449733,7,10.0,7.0,7.0
0,Thur,Lunch,Male,Yes,3.058000,1.235596,10,13.0,6.0,8.0
30,Sun,Dinner,Female,Yes,3.500000,0.166667,4,19.0,2.0,9.0
22,Sat,Dinner,Female,Yes,2.868667,2.135627,15,7.0,14.0,9.0


In [ ]:
#stable 함수

In [102]:
penguins = sns.load_dataset("penguins")
penguins

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female
...,...,...,...,...,...,...,...
339,Gentoo,Biscoe,NaN,NaN,NaN,NaN,NaN
340,Gentoo,Biscoe,46.8,14.3,215.0,4850.0,Female
341,Gentoo,Biscoe,50.4,15.7,222.0,5750.0,Male
342,Gentoo,Biscoe,45.2,14.8,212.0,5200.0,Female


In [124]:
penguins['weight'] = penguins['bill_length_mm'] * penguins['bill_depth_mm']
penguins

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,body_mass_kg,weight
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male,3.75,731.17
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female,3.80,687.30
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female,3.25,725.40
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female,3.45,708.31
...,...,...,...,...,...,...,...,...,...
339,Gentoo,Biscoe,NaN,NaN,NaN,NaN,NaN,NaN,NaN
340,Gentoo,Biscoe,46.8,14.3,215.0,4850.0,Female,4.85,669.24
341,Gentoo,Biscoe,50.4,15.7,222.0,5750.0,Male,5.75,791.28
342,Gentoo,Biscoe,45.2,14.8,212.0,5200.0,Female,5.20,668.96


In [ ]:
#1번
penguins['body_mass_kg'] = penguins['body_mass_g'] / 1000
result = penguins.groupby(['sex','species'])['body_mass_kg'].mean().sort_values(ascending=False)
result2 = penguins[['sex','species','bill_length_mm','bill_depth_mm','body_mass_kg']].sort_values('body_mass_kg',ascending=False)
result3 = penguins[['sex','species','bill_length_mm','bill_depth_mm']].sort_values('bill_depth_mm',ascending=False)
result4 = penguins[['sex','species','bill_length_mm','bill_depth_mm']].sort_values('bill_length_mm',ascending=False)
print(result)
print(result2.head(5))
print(result3.head(5))
print(result4.head(5))
#gentoo 종이 대체로 무겁다. 부리 길이는 무게에 큰 영향은 없는것 같은데 부리두께의 경우는 무게에 어느정도 영향이 있는것 같다.

sex     species  
Male    Gentoo       5.484836
Female  Gentoo       4.679741
Male    Adelie       4.043493
        Chinstrap    3.938971
Female  Chinstrap    3.527206
        Adelie       3.368836
Name: body_mass_kg, dtype: float64
      sex species  bill_length_mm  bill_depth_mm  body_mass_kg
237  Male  Gentoo            49.2           15.2          6.30
253  Male  Gentoo            59.6           17.0          6.05
337  Male  Gentoo            48.8           16.2          6.00
297  Male  Gentoo            51.1           16.3          6.00
299  Male  Gentoo            45.2           16.4          5.95
     sex species  bill_length_mm  bill_depth_mm
19  Male  Adelie            46.0           21.5
13  Male  Adelie            38.6           21.2
49  Male  Adelie            42.3           21.2
35  Male  Adelie            39.2           21.1
14  Male  Adelie            34.6           21.1
        sex    species  bill_length_mm  bill_depth_mm
253    Male     Gentoo            59.6         

In [ ]:
#2번
penguins['flipper_rank'] = penguins['flipper_length_mm'].rank(pct=True)
bins = [0, 0.2, 0.5, 1.0]
labels = ["short","mid",'long']
penguins['flipper_level'] = pd.cut(penguins['flipper_rank'],bins=bins,labels=labels)
penguins.groupby(['species','island','flipper_level'],observed=False)['body_mass_g'].mean().sort_values(ascending=False).head(3)
# penguins.sort_values('flipper_length_mm',ascending=False)
#biscoe 지역에 사는 펭귄들이 대체로 무겁고 날개가 길다

species  island  flipper_level
Gentoo   Biscoe  long             5076.016260
Adelie   Biscoe  long             4233.333333
         Dream   long             4100.000000
Name: body_mass_g, dtype: float64

In [ ]:
#3번
total = penguins.groupby(['species','island','sex','flipper_level'],observed=False)['body_mass_g'].agg(['mean','var','count']).reset_index()
total['score'] = total['mean'] / (total['var'] + 0.0001)
total.sort_values(['score'],ascending=False).head(7)

,species,island,sex,flipper_level,mean,var,count,score
2,Adelie,Biscoe,Female,long,3787.500000,2812.500000,2,1.346667
15,Adelie,Torgersen,Male,short,3825.000000,11250.000000,2,0.340000
7,Adelie,Dream,Female,mid,3404.545455,37352.272727,11,0.091147
11,Adelie,Dream,Male,long,4210.000000,46437.500000,5,0.090659
13,Adelie,Torgersen,Female,mid,3380.769231,47828.525641,13,0.070685
6,Adelie,Dream,Female,short,3286.666667,47398.809524,15,0.069341
38,Gentoo,Biscoe,Female,long,4679.741379,79286.335451,58,0.059023


In [ ]:
total.sort_values(['mean'],ascending=False).head(7)
#젠투 펭귄은 비스코 지역에서만 서식하는데 이 젠투 펭귄 종 자체가 크기가 크고 무거움
#펭귄은 수컷이 암컷보다 무겁다
#근데 분산 무게의 평균보다 영향이 커서 정확도가 떨어지는것 같음
#날개가 긴 개체들이 대체로 무거움

,species,island,sex,flipper_level,mean,var,count,score
41,Gentoo,Biscoe,Male,long,5484.836066,98068.306011,61,0.055929
38,Gentoo,Biscoe,Female,long,4679.741379,79286.335451,58,0.059023
5,Adelie,Biscoe,Male,long,4456.250000,123906.250000,4,0.035965
11,Adelie,Dream,Male,long,4210.000000,46437.500000,5,0.090659
16,Adelie,Torgersen,Male,mid,4090.909091,171534.090909,11,0.023849
10,Adelie,Dream,Male,mid,4048.437500,79372.395833,16,0.051006
4,Adelie,Biscoe,Male,mid,4047.500000,85895.833333,10,0.047121


In [155]:
print(penguins['species'].unique())

['Adelie' 'Chinstrap' 'Gentoo']


In [170]:
flights = sns.load_dataset("flights")
flights

,year,month,passengers
0,1949,Jan,112
1,1949,Feb,118
2,1949,Mar,132
3,1949,Apr,129
4,1949,May,121
...,...,...,...
139,1960,Aug,606
140,1960,Sep,508
141,1960,Oct,461
142,1960,Nov,390


In [ ]:
#1번
print(flights.groupby('year',observed=False)['passengers'].agg(['sum','mean']))
result = flights.groupby('month',observed=False)['passengers'].agg(['sum','mean'])
print(result.sort_values('mean',ascending=False))
#과학기술이 발전함에 따라 세계회가 진행되며 비행기를 타는 승객수가 점진적으로 증가하고 있다
#역시 여름에 다들 해외로 놀러간다. 7월 goat

       sum        mean
year                  
1949  1520  126.666667
1950  1676  139.666667
1951  2042  170.166667
1952  2364  197.000000
1953  2700  225.000000
1954  2867  238.916667
1955  3408  284.000000
1956  3939  328.250000
1957  4421  368.416667
1958  4572  381.000000
1959  5140  428.333333
1960  5714  476.166667
        sum        mean
month                  
Jul    4216  351.333333
Aug    4213  351.083333
Jun    3740  311.666667
Sep    3629  302.416667
May    3262  271.833333
Mar    3242  270.166667
Apr    3205  267.083333
Oct    3199  266.583333
Dec    3142  261.833333
Jan    2901  241.750000
Feb    2820  235.000000
Nov    2794  232.833333
        sum        mean
month                  
Jan    2901  241.750000
Feb    2820  235.000000
Mar    3242  270.166667
Apr    3205  267.083333
May    3262  271.833333
Jun    3740  311.666667
Jul    4216  351.333333
Aug    4213  351.083333
Sep    3629  302.416667
Oct    3199  266.583333
Nov    2794  232.833333
Dec    3142  261.833333


In [ ]:
#2번
flights['flights_rank'] = flights['passengers'].rank(pct=True)
bins = [0, 0.2, 0.5, 1.0]
labels = ["low-traffic","mid-traffic",'high-traffic']
flights['traffic_level'] = pd.cut(flights['flights_rank'],bins=bins,labels=labels)
flights.groupby(['month','traffic_level'],observed=False)['passengers'].mean().rank(ascending=False).sort_values().head(3)
#7,8,6 순서가 1등 2등 3등이므로 여름철 휴가기간에 비행기를 제일 많이 탑승한다.

month  traffic_level
Jul    high-traffic     1.0
Aug    high-traffic     2.0
Jun    high-traffic     3.0
Name: passengers, dtype: float64

In [194]:
flights['month']

0      Jan
1      Feb
2      Mar
3      Apr
4      May
      ... 
139    Aug
140    Sep
141    Oct
142    Nov
143    Dec
Name: month, Length: 144, dtype: category
Categories (12, object): ['Jan', 'Feb', 'Mar', 'Apr', ..., 'Sep', 'Oct', 'Nov', 'Dec']

In [195]:
def season(month):
    if month in 'Jan, Feb, Dec':
        return 'winter'
    elif month in 'Mar, Apr, May':
        return 'spring'
    elif month in 'Jun, Jul, Aug':
        return 'summer'
    else:
        return 'fall'

In [200]:
def period(year):
    if year <= 1952:
        return 'early'
    else:
        return 'late'

In [ ]:
flights['season']=flights['month'].apply(season)
flights['period']=flights['year'].apply(period)
result = flights.groupby(['season','period'])['passengers'].agg(['mean','var','std','count'])
result['Stable Peak Season'] = result['mean'] / (result['std']/result['mean'] + 0.1)
result.sort_values('Stable Peak Season',ascending=False)
#표준편차를 mean으로 나누어 데이터 스케일에 대한 차이를 줄여줌(분모 크기 낮추기)
#기간은 비교적 최근일수록 여름에 압도적으로 비행기를 많이 탄다.

NameError: name 'flights' is not defined